# 1. Setup & Imports
This section imports required libraries, configures paths, and defines the experiment scope for FinSight XGBoost forecasting.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "HDFCBANK.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: HDFCBANK.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,639.500000,644.000000,639.500000,643.375000,6137166,609.389160,0.006374,0.006354,4.500000
2020-01-03,641.099976,642.500000,631.799988,634.200012,10855550,600.698792,-0.014261,-0.014363,10.700012
2020-01-06,630.000000,630.900024,618.000000,620.474976,10890186,587.698853,-0.021641,-0.021879,12.900024
2020-01-07,629.450012,635.724976,626.125000,630.299988,14724494,597.004822,0.015835,0.015711,9.599976
2020-01-08,623.474976,631.075012,620.025024,628.650024,11332110,595.441956,-0.002618,-0.002621,11.049988


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-1.277790,-1.271400,-1.224250,-1.233929,-0.545518,-1.240528,0.617558,0.620802,-0.608115,-1.282271,-1.207101,-1.030540,-1.141682,-0.666851,-1.272539
2020-01-30,-1.223918,-1.283469,-1.244035,-1.272539,-0.617486,-1.275065,-0.504253,-0.496689,-0.470182,-1.232459,-1.191381,-1.017203,-1.153743,-0.654015,-1.271554
2020-01-31,-1.253518,-1.288019,-1.232086,-1.271554,-0.657571,-1.274184,-0.004874,0.003287,-0.759839,-1.271048,-1.192560,-1.054468,-1.162108,-0.614829,-1.403536
2020-02-03,-1.389483,-1.445708,-1.398789,-1.403536,-0.576056,-1.392246,-1.694632,-1.705219,-0.573632,-1.270064,-1.315766,-1.145670,-1.171675,-0.434574,-1.257765
2020-02-04,-1.385536,-1.303056,-1.319257,-1.257765,-0.240307,-1.261849,1.887106,1.861340,0.512595,-1.401976,-1.276466,-1.187054,-1.177795,-0.416229,-1.199259


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split with the last 20% as test data (no shuffling).

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror",
        tree_method="hist",
        device="cuda"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

[0]	validation_0-rmse:1.56331
[100]	validation_0-rmse:0.50011
[200]	validation_0-rmse:0.49275
[300]	validation_0-rmse:0.49397
[400]	validation_0-rmse:0.49433
[499]	validation_0-rmse:0.49442
Fold 1 RMSE: 0.494422
[0]	validation_0-rmse:0.81317


c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\xgboost\core.py:751: UserWarning: [20:55:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[100]	validation_0-rmse:0.18242
[200]	validation_0-rmse:0.18382
[300]	validation_0-rmse:0.18437
[400]	validation_0-rmse:0.18415
[499]	validation_0-rmse:0.18441
Fold 2 RMSE: 0.184415
[0]	validation_0-rmse:0.88158
[100]	validation_0-rmse:0.11595
[200]	validation_0-rmse:0.11295
[300]	validation_0-rmse:0.11349
[400]	validation_0-rmse:0.11337
[499]	validation_0-rmse:0.11357
Fold 3 RMSE: 0.113574
[0]	validation_0-rmse:0.75363
[100]	validation_0-rmse:0.08497
[200]	validation_0-rmse:0.08991
[300]	validation_0-rmse:0.09150
[400]	validation_0-rmse:0.09202
[499]	validation_0-rmse:0.09251
Fold 4 RMSE: 0.092511
[0]	validation_0-rmse:0.88908
[100]	validation_0-rmse:0.26537
[200]	validation_0-rmse:0.26085
[300]	validation_0-rmse:0.26352
[400]	validation_0-rmse:0.26479
[499]	validation_0-rmse:0.26504
Fold 5 RMSE: 0.265039
Mean CV RMSE: 0.229992


# 6. Hyperparameter Tuning (Optuna)
Tune XGBoost hyperparameters with 50 Optuna trials using time-series CV and GPU execution.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda"
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-24 20:56:10,094] A new study created in memory with name: no-name-346b4044-6e8e-4250-bb1f-d9fe47260578


[0]	validation_0-rmse:0.89366
[100]	validation_0-rmse:0.15255
[200]	validation_0-rmse:0.15200
[300]	validation_0-rmse:0.15237
[332]	validation_0-rmse:0.15242
[0]	validation_0-rmse:0.81647
[100]	validation_0-rmse:0.09518
[200]	validation_0-rmse:0.09687
[300]	validation_0-rmse:0.09706
[332]	validation_0-rmse:0.09736
[0]	validation_0-rmse:0.71852
[100]	validation_0-rmse:0.20685
[200]	validation_0-rmse:0.20208
[300]	validation_0-rmse:0.20345
[332]	validation_0-rmse:0.20272


[I 2026-05-24 20:56:57,801] Trial 0 finished with value: 0.15083413864268513 and parameters: {'n_estimators': 333, 'learning_rate': 0.20399829315642912, 'max_depth': 5, 'subsample': 0.6773134255195675, 'colsample_bytree': 0.7220288818375744, 'min_child_weight': 7}. Best is trial 0 with value: 0.15083413864268513.


Trial 0 | RMSE: 0.1508 | params: {'n_estimators': 333, 'learning_rate': 0.20399829315642912, 'max_depth': 5, 'subsample': 0.6773134255195675, 'colsample_bytree': 0.7220288818375744, 'min_child_weight': 7}
[0]	validation_0-rmse:1.02346
[100]	validation_0-rmse:0.14073
[200]	validation_0-rmse:0.14191
[300]	validation_0-rmse:0.14182
[353]	validation_0-rmse:0.14181
[0]	validation_0-rmse:0.92144
[100]	validation_0-rmse:0.10893
[200]	validation_0-rmse:0.10908
[300]	validation_0-rmse:0.10924
[353]	validation_0-rmse:0.10928
[0]	validation_0-rmse:0.79979
[100]	validation_0-rmse:0.22413
[200]	validation_0-rmse:0.22602
[300]	validation_0-rmse:0.22617
[353]	validation_0-rmse:0.22619
Trial 1 | RMSE: 0.1591 | params: {'n_estimators': 354, 'learning_rate': 0.08096101432536228, 'max_depth': 10, 'subsample': 0.7857682471063876, 'colsample_bytree': 0.8870003147717049, 'min_child_weight': 2}


[I 2026-05-24 20:58:23,829] Trial 1 finished with value: 0.15909174573431215 and parameters: {'n_estimators': 354, 'learning_rate': 0.08096101432536228, 'max_depth': 10, 'subsample': 0.7857682471063876, 'colsample_bytree': 0.8870003147717049, 'min_child_weight': 2}. Best is trial 0 with value: 0.15083413864268513.


...

Trial 17 | RMSE: 0.1499 | params: {'n_estimators': 648, 'learning_rate': 0.046412699594620924, 'max_depth': 9, 'subsample': 0.7438417022597776, 'colsample_bytree': 0.6820945485105195, 'min_child_weight': 8}
[0]	validation_0-rmse:0.96626
[100]	validation_0-rmse:0.15350
[200]	validation_0-rmse:0.15387
[300]	validation_0-rmse:0.15384
[400]	validation_0-rmse:0.15384
[401]	validation_0-rmse:0.15384
[0]	validation_0-rmse:0.87562
[100]	validation_0-rmse:0.09157
[200]	validation_0-rmse:0.09303
[300]	validation_0-rmse:0.09314
[400]	validation_0-rmse:0.09313
[401]	validation_0-rmse:0.09313
[0]	validation_0-rmse:0.76411
[100]	validation_0-rmse:0.19722
[200]	validation_0-rmse:0.19896
[300]	validation_0-rmse:0.19922
[400]	validation_0-rmse:0.19920
[401]	validation_0-rmse:0.19922


[I 2026-05-24 21:22:14,618] Trial 18 finished with value: 0.14873322009008524 and parameters: {'n_estimators': 402, 'learning_rate': 0.13409020006888517, 'max_depth': 9, 'subsample': 0.8566996754794194, 'colsample_bytree': 0.9700673291406763, 'min_child_weight': 6}. Best is trial 14 with value: 0.1458241188188727.


Trial 18 | RMSE: 0.1487 | params: {'n_estimators': 402, 'learning_rate': 0.13409020006888517, 'max_depth': 9, 'subsample': 0.8566996754794194, 'colsample_bytree': 0.9700673291406763, 'min_child_weight': 6}
[0]	validation_0-rmse:1.08644
[100]	validation_0-rmse:0.20190
[200]	validation_0-rmse:0.14908
[274]	validation_0-rmse:0.14757
[0]	validation_0-rmse:0.97300
[100]	validation_0-rmse:0.16086
[200]	validation_0-rmse:0.09223
[274]	validation_0-rmse:0.09047
[0]	validation_0-rmse:0.83918
[100]	validation_0-rmse:0.26365
[200]	validation_0-rmse:0.21106
[274]	validation_0-rmse:0.20307


[I 2026-05-24 21:22:59,208] Trial 19 finished with value: 0.14703778530863196 and parameters: {'n_estimators': 275, 'learning_rate': 0.02171831802347411, 'max_depth': 7, 'subsample': 0.8686994908011865, 'colsample_bytree': 0.8801297772476299, 'min_child_weight': 8}. Best is trial 14 with value: 0.1458241188188727.


Trial 19 | RMSE: 0.1470 | params: {'n_estimators': 275, 'learning_rate': 0.02171831802347411, 'max_depth': 7, 'subsample': 0.8686994908011865, 'colsample_bytree': 0.8801297772476299, 'min_child_weight': 8}
Best RMSE: 0.1458241188188727
Best params:
  n_estimators: 585
  learning_rate: 0.014151024758081159
  max_depth: 8
  subsample: 0.8587279113760851
  colsample_bytree: 0.8910281877254501
  min_child_weight: 8


# 7. Final Model Training
Train the final model with best Optuna parameters on the full training set and evaluate on the holdout test set.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "device": "cuda"
})

final_model = XGBRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print(f"Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full, verbose=100)
print("Final model trained on full dataset")



# AUTO SAVE immediately after training
from pathlib import Path
import joblib, json
_save_dir = Path("./")
joblib.dump(final_model, _save_dir / "xgboost_model.pkl")
with open(_save_dir / "xgboost_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
with open(_save_dir / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)
print("AUTO SAVED")

[0]	validation_0-rmse:1.61718
[100]	validation_0-rmse:0.70026
[200]	validation_0-rmse:0.45400
[300]	validation_0-rmse:0.41089
[400]	validation_0-rmse:0.40230
[500]	validation_0-rmse:0.40159
[584]	validation_0-rmse:0.40314
RMSE: 0.403140
MAE:  0.327727
MAPE: 34.5491%
Retraining on full dataset...
Final model trained on full dataset
AUTO SAVED


In [8]:
import importlib
import nbformat
importlib.reload(nbformat)
print(nbformat.__version__)

5.10.4


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained XGBoost model.

In [9]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} XGBoost Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
fig_imp.show()

# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [10]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

# Inverse transform to actual prices
try:
    y_test_actual = preprocessor.inverse_transform(y_test.values)
    test_pred_actual = preprocessor.inverse_transform(test_pred)
    future_pred_actual = preprocessor.inverse_transform(future_pred)
except:
    y_test_actual = y_test.values
    test_pred_actual = test_pred
    future_pred_actual = future_pred

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_test_actual, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=test_pred_actual, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred_actual, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    height=600
)
fig_fc.show()

# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the ticker-specific artifacts directory.

In [11]:
model_path = ARTIFACTS_DIR / "xgboost_model.pkl"
feature_cols_path = ARTIFACTS_DIR / "feature_columns.json"
best_params_path = ARTIFACTS_DIR / "xgboost_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\xgboost_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\xgboost_best_params.json
